In [1]:
from pathlib import Path
import sys

# Resolve paths even if the working directory is repo root.
notebook_dir = Path.cwd()
if not (notebook_dir / "conf.toml").exists():
    notebook_dir = notebook_dir / "notebook"
repo_root = notebook_dir.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

paths = {
    "conf": notebook_dir / "conf.toml",
    "doc": notebook_dir / "blob_documentation.txt",
    "code": notebook_dir / "blob.js",
    "spec": notebook_dir / "blob.webidl",
}

In [2]:
from circinus.settings import load_config

In [3]:
from circinus.agent import Agent

In [4]:
config = load_config(str(paths["conf"]))

In [5]:
agent = Agent(config)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [6]:
snippets = agent.fuzz(
    documentation=str(paths["doc"]),
    code=str(paths["code"]),
    specification=str(paths["spec"]),
)

In [7]:
# Verify mock mode config
from circinus.settings import settings
print(f"Mode: {settings.mode}")
print(f"Config: {settings.to_dict()}")

Mode: mock
Config: {'OPENAI_API_KEY': '${OPENAI_API_KEY}', 'OPENAI_API_MODEL': 'gpt-3.5-turbo', 'MAX_TOKENS': 2048, 'MODE': 'mock'}


In [8]:
from pathlib import Path

# Create simple demo files
demo_dir = Path("demo_files")
demo_dir.mkdir(exist_ok=True)

# Demo Python code
(demo_dir / "demo.py").write_text("""
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b
""")

# Demo documentation
(demo_dir / "doc.txt").write_text("Simple math functions: add and multiply")

# Demo specification
(demo_dir / "spec.txt").write_text("Functions should handle edge cases")

# Now fuzz the demo files
snippets = agent.fuzz(
    documentation=str(demo_dir / "doc.txt"),
    code=str(demo_dir / "demo.py"),
    specification=str(demo_dir / "spec.txt"),
)

# Print first 5 snippets
for i, snippet in enumerate(snippets):
    if i >= 5:
        break
    print(f"--- Snippet {i+1} ---")
    print(snippet)

--- Snippet 1 ---
def mock_function(x, y):
    return x + y
--- Snippet 2 ---
def mock_function(x, y):
    return x + y
--- Snippet 3 ---
def mock_function(x, y):
    return x + y
--- Snippet 4 ---
def mock_function(x, y):
    return x + y
--- Snippet 5 ---
def mock_function(x, y):
    return x + y


In [9]:
from itertools import islice

# Status check: generate one snippet to confirm output
snippets_preview = list(islice(agent.fuzz(
    documentation=str(demo_dir / "doc.txt"),
    code=str(demo_dir / "demo.py"),
    specification=str(demo_dir / "spec.txt"),
), 1))

print(f"Preview count: {len(snippets_preview)}")
if snippets_preview:
    print(snippets_preview[0])

Preview count: 1
def mock_function(x, y):
    return x + y


In [10]:
for snippet in snippets:
    print(snippet)

def mock_function(x, y):
    return x + y
def mock_function(x, y):
    return x + y
def mock_function(x, y):
    return x + y
